# PatchTCN — 端到端量价时序预测

**提交清单（严格 4 个文件）**
- `submit.ipynb`（本文件，平台入口）
- `train.py`（可从官方原始数据重新拟合标准化并从零重训）
- `predict.py`（推理，仅输出 date / instrument / score）
- `model_weights.json`（公榜权重，< 50MB）

**模型**：因果空洞卷积（6 层，dilation 1→32），输入为 1 分钟原始字段构成的日内 patch 序列，
回看 5 个交易日。无循环结构、无外部预训练权重。

**输入合规**：仅使用主办方原始字段（25 个，上限 100）；预处理只含缺失填充、按字段固定单位换算、
log1p，以及仅在训练区间拟合的标准化。未使用收益率、价差、盘口不平衡、滚动统计、技术指标或降维。

In [ ]:
import os

MODEL_FILE = "model_weights.json"
MODEL_PATH = os.path.join(os.getcwd(), MODEL_FILE)

from predict import main as _predict


def main(datasources, start_date, end_date):
    """平台入口：返回仅含 date / instrument / score 三列的 DataFrame。"""
    return _predict(datasources, start_date, end_date, model_path=MODEL_PATH)

## 私榜重训入口

私榜阶段平台在隔离环境按训练脚本从零重训。下面的调用展示了重训接口；
`train_and_save` 会自行按短日期块拉取原始数据、重新拟合标准化并写出权重。

In [ ]:
# 平台重训时执行（公榜推理不需要）
# from train import train_and_save, TABLE_CLOUD, TRAIN_START, TRAIN_END
#
# train_and_save(
#     table=TABLE_CLOUD,
#     save_path=MODEL_PATH,
#     start=TRAIN_START,
#     end=TRAIN_END,
#     is_local=False,
# )